# Visualisation & story — European commercial airspace

Reads the result parquets produced by `04_spark_etl.ipynb` (in `../data/processed/`) and turns them into the charts that tell the story. **No Kafka / Spark needed** — this works purely on the stored results.

- **Q1** Dominance — top airlines & carrier nations
- **Q2** Geography — flight-density map + whose airspace is busiest
- **Q3** Busiest airports
- **Q4a** Top departure vs arrival airports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "../data/processed"   # results live in the repo-root data/processed

by_airline   = pd.read_parquet(f"{DATA_DIR}/dominance_by_airline.parquet")
by_country   = pd.read_parquet(f"{DATA_DIR}/dominance_by_country.parquet")
density      = pd.read_parquet(f"{DATA_DIR}/density_grid.parquet")
busiest      = pd.read_parquet(f"{DATA_DIR}/busiest_airports.parquet")
departures   = pd.read_parquet(f"{DATA_DIR}/top_departures.parquet")
arrivals     = pd.read_parquet(f"{DATA_DIR}/top_arrivals.parquet")

print("loaded:", {k: len(v) for k, v in {
    'airlines': by_airline, 'countries': by_country, 'grid': density,
    'airports': busiest, 'departures': departures, 'arrivals': arrivals}.items()})

## Q1 — Dominance: who rules the European sky?

In [ ]:
top = by_airline.head(12).iloc[::-1]   # reverse so the biggest is on top in barh
plt.figure(figsize=(9, 5))
plt.barh(top["airline"], top["aircraft"], color="#2b8cbe")
plt.xlabel("distinct aircraft aloft")
plt.title("Top airlines over Europe (commercial)")
plt.tight_layout()
plt.show()

In [ ]:
topc = by_country.head(12).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh(topc["origin_country"], topc["aircraft"], color="#41ab5d")
plt.xlabel("distinct aircraft aloft")
plt.title("Top carrier nations over Europe (by operator country)")
plt.tight_layout()
plt.show()

## Q2 — Geography: the shape of Europe's air traffic
Each point is a ~0.1° grid cell coloured by how many flight records passed through it. The densest cells trace the busiest **corridors** — the map basically draws itself from the traffic.

In [ ]:
plt.figure(figsize=(10, 9))
sc = plt.scatter(density["lon_bin"], density["lat_bin"],
                 c=density["flights"], cmap="inferno", s=6, norm="log")
plt.colorbar(sc, label="flight records (log scale)")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.title("European commercial air-traffic density")
plt.gca().set_aspect(1.4)   # rough aspect correction for European latitudes
plt.tight_layout()
plt.show()

### Whose airspace is busiest?
We reverse-geocode each grid cell to a country and sum the flights. Needs `reverse_geocoder` (offline): `pip install reverse_geocoder`.

In [ ]:
try:
    import reverse_geocoder as rg
    coords = list(zip(density["lat_bin"].tolist(), density["lon_bin"].tolist()))
    hits = rg.search(coords, mode=1)          # mode=1 = single-threaded (notebook-safe)
    density["cc"] = [h["cc"] for h in hits]    # ISO-2 country code
    airspace = (density.groupby("cc")["flights"].sum()
                .sort_values(ascending=False).head(12).iloc[::-1])

    plt.figure(figsize=(9, 5))
    plt.barh(airspace.index, airspace.values, color="#d7301f")
    plt.xlabel("flight records over the country's territory")
    plt.title("Busiest airspace by country (overflights)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("reverse_geocoder not installed -> run: pip install reverse_geocoder")

## Q3 — Busiest airports (reconstructed from live positions)
Inferred purely from low climbing/descending aircraft matched to their nearest airport.

In [ ]:
topa = busiest.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(topa["airport_name"], topa["aircraft"], color="#6a51a3")
plt.xlabel("distinct aircraft near the airport")
plt.title("Busiest European airports (inferred from live traffic)")
plt.tight_layout()
plt.show()

## Q4a — Departures vs arrivals
Same airport matches, split by flight phase (climbing = departing, descending = arriving).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

d = departures.head(10).iloc[::-1]
axes[0].barh(d["airport_name"], d["departures"], color="#3182bd")
axes[0].set_title("Top departure airports")
axes[0].set_xlabel("distinct departing aircraft")

a = arrivals.head(10).iloc[::-1]
axes[1].barh(a["airport_name"], a["arrivals"], color="#e6550d")
axes[1].set_title("Top arrival airports")
axes[1].set_xlabel("distinct arriving aircraft")

plt.tight_layout()
plt.show()

## Story / takeaways
- **Who owns the European sky?** A handful of low-cost & legacy carriers dominate — Ryanair far ahead, then Turkish, easyJet, Lufthansa, Air France…
- **Where?** The density map traces the classic core corridors (UK – Benelux – Germany – N. Italy) without us drawing a single border.
- **Busiest hubs** reconstructed from raw positions match the real-world ranking (Istanbul, Heathrow, Frankfurt, Schiphol, CDG…) — i.e. live ADS-B data alone recovers reality.
- **Departures vs arrivals** line up with known hub roles.

_Note: numbers reflect the snapshot window captured in Kafka at ETL time; rerunning `04` with more accumulated data sharpens them._